# 🟢 PaddleOCR LOCAL FAST V7 · ROBUST RESUME

Corrige la detección del runtime local.

Ya **no depende de que el repo esté montado en `/content/work`**.
Detecta el entorno local por:
- Python 3.12;
- kernel WSL2;
- GPU visible.

Los archivos temporales, la caché de pip y los modelos viven bajo `/root`,
que en tu configuración Docker ya es persistente.

Versiones:
- PaddlePaddle GPU 3.2.0
- CUDA wheel cu126
- PaddleOCR 3.2.0

No usa venv y no reinicia el kernel.

Durante la instalación muestra:
- barra de descarga real;
- MB descargados / total;
- porcentaje;
- velocidad estimada por `tqdm`;
- consola de `pip`;
- heartbeat si `pip` queda silencioso.

### Cambio importante en V5

El wheel gigante de Paddle **ya no lo descarga pip**.

Se guarda persistentemente en:

`/root/.cache/paddle-wheels/`

La descarga usa `curl --continue-at -`, por lo que si falla a mitad de camino,
la próxima ejecución **reanuda** el mismo archivo en lugar de empezar desde cero.

Antes de instalar se valida que el `.whl` sea un ZIP/wheel íntegro.


### V6: progreso independiente de curl

Jupyter a veces no muestra las barras `\r` de curl.

V6 mide directamente cada 0,5 segundos cuánto crece el archivo `.whl` y muestra,
cada ~2 segundos, una línea normal como:

`812.4 MB / 1.88 GB (42.2%) · 6.4 MB/s · ETA 2.8 min`

Por eso hay feedback aunque curl permanezca totalmente silencioso.


### V7: tamaño remoto + validación en Linux

V7 ya no considera que `curl` haya terminado solo porque devolvió código 0.

1. consulta el tamaño remoto real mediante HTTP Range;
2. compara bytes locales vs remotos;
3. reanuda hasta que ambos sean exactamente iguales;
4. conserva el wheel persistente en `/root/.cache/paddle-wheels`;
5. copia el wheel completo a `/tmp`;
6. valida e instala desde `/tmp`, evitando seeks ZIP sobre el bind mount de Windows.


In [ ]:
import sys, platform, subprocess, importlib.metadata as md, socket, pathlib

print("🟢 LOCAL FAST V3 · Paddle 3.2.0 / PaddleOCR 3.2.0")
print()

release = platform.release()
platform_text = platform.platform()
host = socket.gethostname()

print("Python:", sys.version)
print("Platform:", platform_text)
print("Kernel:", release)
print("Hostname:", host)
print("HOME:", pathlib.Path.home())
print()

# El runtime Docker local corre sobre WSL2.
is_wsl = ("microsoft" in release.lower()) or ("wsl" in release.lower())
is_py312 = sys.version_info[:2] == (3, 12)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_text = gpu.stdout.strip()

print("GPU:", gpu_text or "No detectada")
print()

if not is_py312:
    raise RuntimeError(
        f"Este notebook espera Python 3.12 del runtime local. "
        f"Encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

if not is_wsl:
    print("⚠️ El kernel no parece WSL2.")
    print("Esto podría ser Colab Cloud. Revisá que estés conectado al runtime local.")
    print("No voy a instalar nada automáticamente en esta celda.")
else:
    print("✅ Runtime WSL2 local detectado.")

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "NO INSTALADO"

print()
print("=== PAQUETES ===")
for name in ["paddlepaddle-gpu","paddleocr","paddlex","torch","pillow","numpy"]:
    print(f"{name:20} {ver(name)}")

In [ ]:
import sys, os, pathlib, subprocess, importlib.metadata as md, platform
import shutil, time, re

PADDLE = "3.2.0"
OCR = "3.2.0"

PADDLE_WHEEL_URL = (
    "https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/"
    "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"
)

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError(
        "ABORTADO: este kernel no parece WSL2/local. "
        "No voy a descargar Paddle por accidente en Colab Cloud."
    )

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Este wheel requiere Python 3.12. "
        f"Encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

HOME = pathlib.Path.home()
PIP_CACHE = HOME / ".cache" / "pip"
MODEL_CACHE = HOME / ".cache" / "paddlex"
WHEEL_CACHE = HOME / ".cache" / "paddle-wheels"

for d in (PIP_CACHE, MODEL_CACHE, WHEEL_CACHE):
    d.mkdir(parents=True, exist_ok=True)

os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE)
os.environ["PADDLE_PDX_CACHE_HOME"] = str(MODEL_CACHE)
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

WHEEL_NAME = "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"
WHEEL = WHEEL_CACHE / WHEEL_NAME
TMP_WHEEL = pathlib.Path("/tmp") / WHEEL_NAME

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

def human_bytes(n):
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while n >= 1024 and i < len(units)-1:
        n /= 1024
        i += 1
    return f"{n:.2f} {units[i]}"

def run(cmd, **kwargs):
    print("$", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

def remote_size_via_range(url):
    """
    Pide solo el primer byte. Un servidor con Range responde:
      Content-Range: bytes 0-0/TOTAL
    Eso nos da el tamaño real sin descargar el wheel.
    """
    curl = shutil.which("curl")
    p = subprocess.run(
        [
            curl,
            "--location",
            "--silent",
            "--show-error",
            "--fail",
            "--range", "0-0",
            "--dump-header", "-",
            "--output", "/dev/null",
            url,
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    total = None
    for line in p.stdout.splitlines():
        m = re.search(r"content-range:\s*bytes\s+\d+-\d+/(\d+)", line, re.I)
        if m:
            total = int(m.group(1))

    if total is None:
        # fallback a Content-Length del último bloque
        lengths = []
        for line in p.stdout.splitlines():
            m = re.search(r"content-length:\s*(\d+)", line, re.I)
            if m:
                lengths.append(int(m.group(1)))
        if lengths:
            total = lengths[-1]

    if not total:
        raise RuntimeError(
            "No pude determinar el tamaño remoto del wheel. "
            "No voy a asumir que una descarga parcial está completa."
        )

    return total

def download_until_complete(url, destination, total):
    """
    Reanuda tantas veces como sea necesario hasta que size == total.
    El progreso se calcula mirando el archivo, no la salida de curl.
    """
    curl = shutil.which("curl")
    destination = pathlib.Path(destination)

    if destination.exists() and destination.stat().st_size > total:
        oversize = destination.with_suffix(destination.suffix + ".oversize")
        print(
            f"⚠️ El archivo local ({human_bytes(destination.stat().st_size)}) "
            f"es mayor al remoto ({human_bytes(total)})."
        )
        print("Lo renombro para no destruirlo:", oversize)
        if oversize.exists():
            oversize.unlink()
        destination.rename(oversize)

    attempt = 0

    while True:
        current = destination.stat().st_size if destination.exists() else 0

        if current == total:
            print("✅ Tamaño local coincide exactamente con el remoto.")
            return

        if current > total:
            raise RuntimeError("El archivo local terminó siendo mayor al remoto.")

        attempt += 1
        print()
        print(f"=== DESCARGA / REANUDACIÓN {attempt} ===")
        print(f"Local:  {human_bytes(current)}")
        print(f"Remoto: {human_bytes(total)}")
        print(f"Falta:  {human_bytes(total-current)}")
        print()

        cmd = [
            curl,
            "--location",
            "--fail",
            "--silent",
            "--show-error",
            "--retry", "20",
            "--retry-delay", "2",
            "--retry-all-errors",
            "--connect-timeout", "30",
            "--continue-at", "-",
            "--output", str(destination),
            url,
        ]

        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            text=True,
        )

        started = time.time()
        last_size = current
        last_t = started
        last_print = 0.0

        while proc.poll() is None:
            now = time.time()
            size = destination.stat().st_size if destination.exists() else 0

            dt = max(0.001, now-last_t)
            dbytes = max(0, size-last_size)
            speed = dbytes/dt

            if now-last_print >= 2:
                pct = min(100.0, size/total*100)
                remaining = max(0, total-size)
                eta = remaining/speed if speed > 0 else None

                if eta is None:
                    eta_txt = "calculando"
                elif eta >= 60:
                    eta_txt = f"{eta/60:.1f} min"
                else:
                    eta_txt = f"{eta:.0f} s"

                print(
                    f"[{now-started:6.1f}s] "
                    f"{human_bytes(size)} / {human_bytes(total)} "
                    f"({pct:5.1f}%) · "
                    f"{human_bytes(speed)}/s · ETA {eta_txt}",
                    flush=True,
                )
                last_print = now

            last_size = size
            last_t = now
            time.sleep(0.5)

        stderr = proc.stderr.read() if proc.stderr else ""
        rc = proc.wait()
        size = destination.stat().st_size if destination.exists() else 0

        if rc != 0:
            print()
            print("curl terminó con código:", rc)
            if stderr.strip():
                print(stderr.strip())
            print("Archivo parcial conservado:", human_bytes(size))
            print("Voy a reintentar/reanudar.")

        # Incluso si curl devolvió 0, NO confiamos en eso:
        # el único criterio de finalización es size == total.
        if size == total:
            print()
            print("✅ Se alcanzó exactamente el tamaño remoto.")
            return

        if size > total:
            raise RuntimeError(
                f"La descarga quedó mayor que el tamaño remoto: "
                f"{size} > {total}"
            )

        print()
        print(
            f"⚠️ La descarga terminó pero aún faltan "
            f"{human_bytes(total-size)}. Reanudando..."
        )

        if attempt >= 30:
            raise RuntimeError(
                "Demasiados intentos de reanudación. "
                "El archivo parcial se conserva."
            )

def copy_to_linux_tmp(src, dst):
    """
    El cache persistente /root puede estar respaldado por un bind mount de Windows.
    Para ZIP/pip usamos /tmp, que es filesystem Linux del contenedor.
    """
    src = pathlib.Path(src)
    dst = pathlib.Path(dst)

    if dst.exists():
        dst.unlink()

    print()
    print("📋 Copiando wheel al filesystem Linux local (/tmp)...")
    print("Origen:", src)
    print("Destino:", dst)
    print("Tamaño:", human_bytes(src.stat().st_size))

    # cp hace lectura secuencial y evita el patrón de seeks de zipfile sobre bind mount.
    run(["cp", str(src), str(dst)])

    if dst.stat().st_size != src.stat().st_size:
        raise RuntimeError("La copia a /tmp terminó con un tamaño diferente.")

    print("✅ Copia completa.")

def validate_wheel_linux(path):
    """
    Primero intenta 'unzip -tqq'. Si unzip no existe, usa Python zipfile
    PERO ya sobre /tmp/ext4, no sobre el bind mount de Windows.
    """
    path = pathlib.Path(path)
    print()
    print("🔎 Validando wheel en /tmp...")

    unzip = shutil.which("unzip")

    if unzip:
        p = subprocess.run(
            [unzip, "-tqq", str(path)],
            capture_output=True,
            text=True,
        )
        if p.returncode != 0:
            detail = (p.stdout + "\n" + p.stderr).strip()
            raise RuntimeError(
                "El wheel falla 'unzip -t' en filesystem Linux.\n"
                + detail[-2000:]
            )
        print("✅ unzip -t: wheel íntegro.")
        return

    print("ℹ️ 'unzip' no está instalado; uso python -m zipfile -t.")
    p = subprocess.run(
        [sys.executable, "-m", "zipfile", "-t", str(path)],
        capture_output=True,
        text=True,
    )
    if p.returncode != 0:
        raise RuntimeError(
            "El wheel falla python -m zipfile -t:\n"
            + (p.stdout + "\n" + p.stderr)[-2000:]
        )

    print("✅ zipfile -t: wheel íntegro.")

print("=== CACHÉS PERSISTENTES ===")
print("pip:     ", PIP_CACHE)
print("modelos: ", MODEL_CACHE)
print("wheel:   ", WHEEL_CACHE)
print()

if ver("paddlepaddle-gpu") == PADDLE:
    print("✅ PaddlePaddle GPU 3.2.0 ya está instalado.")
else:
    existing = ver("paddlepaddle-gpu")
    if existing:
        raise RuntimeError(
            f"Hay PaddlePaddle GPU {existing} instalado. "
            "No voy a mezclar versiones automáticamente."
        )

    print("🌐 Consultando tamaño remoto exacto...")
    remote_total = remote_size_via_range(PADDLE_WHEEL_URL)

    local_size = WHEEL.stat().st_size if WHEEL.exists() else 0

    print("Tamaño remoto:", human_bytes(remote_total), f"({remote_total} bytes)")
    print("Tamaño local: ", human_bytes(local_size), f"({local_size} bytes)")
    print()

    download_until_complete(
        PADDLE_WHEEL_URL,
        WHEEL,
        remote_total,
    )

    # Validar/instalar siempre desde /tmp Linux.
    copy_to_linux_tmp(WHEEL, TMP_WHEEL)
    validate_wheel_linux(TMP_WHEEL)

    print()
    print("=== INSTALANDO PADDLE DESDE /tmp ===")
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "180",
        str(TMP_WHEEL),
    ])

if ver("paddleocr") == OCR:
    print("✅ PaddleOCR 3.2.0 ya está instalado.")
else:
    existing = ver("paddleocr")
    if existing:
        raise RuntimeError(
            f"Hay PaddleOCR {existing} instalado. "
            "No voy a mezclar versiones automáticamente."
        )

    print()
    print("=== INSTALANDO PADDLEOCR ===")
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "180",
        f"paddleocr=={OCR}",
    ])

print()
print("✅ Instalación lista.")
print("NO reinicies el kernel.")
print("La siguiente celda hace el smoke test en un proceso Python nuevo.")

In [ ]:
import sys, subprocess, pathlib, os, platform

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError("No parece el runtime local WSL2.")

dev_dir = pathlib.Path.home()/".cache"/"paddleocr-dev"
dev_dir.mkdir(parents=True, exist_ok=True)

worker_path = dev_dir/"paddle_smoke_worker_v3.py"
worker_path.write_text('\nimport os, sys, json\nfrom pathlib import Path\n\nos.environ.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")\nos.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(Path.home()/".cache"/"paddlex"))\n\nprint("=== WORKER NUEVO ===", flush=True)\nprint("Python:", sys.version, flush=True)\n\nimport paddle\nprint("Paddle:", paddle.__version__, flush=True)\nprint("CUDA:", paddle.is_compiled_with_cuda(), flush=True)\nprint("GPU count:", paddle.device.cuda.device_count(), flush=True)\n\nif not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:\n    raise RuntimeError("Paddle no ve la GPU CUDA.")\n\npaddle.set_device("gpu:0")\ntry:\n    print("GPU:", paddle.device.cuda.get_device_name(), flush=True)\nexcept Exception:\n    print("GPU: gpu:0", flush=True)\n\nimport PIL\nfrom PIL import Image, ImageDraw\nprint("Pillow:", PIL.__version__, flush=True)\n\nfrom paddleocr import PaddleOCR\nimport paddleocr\nprint("PaddleOCR:", getattr(paddleocr, "__version__", "unknown"), flush=True)\n\ndev_dir = Path.home()/".cache"/"paddleocr-dev"\ndev_dir.mkdir(parents=True, exist_ok=True)\n\nimg_path = dev_dir/"paddle_smoke_es.png"\nimg = Image.new("RGB", (1400, 320), "white")\nImageDraw.Draw(img).text(\n    (50, 100),\n    "Histologia epitelio plano simple prueba OCR espanol 12345",\n    fill="black"\n)\nimg.save(img_path)\n\nprint("Imagen:", img_path, flush=True)\nprint("Inicializando OCR...", flush=True)\n\nocr = PaddleOCR(\n    lang="es",\n    device="gpu:0",\n    use_doc_orientation_classify=False,\n    use_doc_unwarping=False,\n    use_textline_orientation=False,\n)\n\nprint("Ejecutando OCR...", flush=True)\nresults = ocr.predict(str(img_path))\n\ntexts = []\nfor res in results:\n    d = getattr(res, "json", res)\n    if callable(d):\n        d = d()\n    if isinstance(d, dict) and "res" in d:\n        d = d["res"]\n    if isinstance(d, dict):\n        texts += [str(x) for x in d.get("rec_texts", [])]\n\nprint("Textos:", json.dumps(texts, ensure_ascii=False), flush=True)\n\nif not texts:\n    raise RuntimeError("No se reconoció texto.")\n\nprint("✅ SMOKE TEST COMPLETO", flush=True)\n', encoding="utf-8")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PADDLE_PDX_MODEL_SOURCE"] = "BOS"
env["PADDLE_PDX_CACHE_HOME"] = str(pathlib.Path.home()/".cache"/"paddlex")

print("Ejecutando worker fresco:")
print(worker_path)
print()

proc = subprocess.Popen(
    [sys.executable, str(worker_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()
if rc:
    raise RuntimeError(f"Worker falló con código {rc}")